In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [4]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [6]:
# Create a single text field by combining the question and options
train_df["text"] = (
    train_df["prompt"].fillna("") + " " +
    train_df["A"].fillna("") + " " +
    train_df["B"].fillna("") + " " +
    train_df["C"].fillna("") + " " +
    train_df["D"].fillna("") + " " +
    train_df["E"].fillna("")
)

test_df["text"] = (
    test_df["prompt"].fillna("") + " " +
    test_df["A"].fillna("") + " " +
    test_df["B"].fillna("") + " " +
    test_df["C"].fillna("") + " " +
    test_df["D"].fillna("") + " " +
    test_df["E"].fillna("")
)

X = train_df["text"]
y = train_df["answer"]

X_test = test_df["text"]

print("Training samples:", len(X))
print("Test samples:", len(X_test))

Training samples: 2000
Test samples: 500


In [7]:
# Split the data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

# Transform the text
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Validation TF-IDF Shape:", X_val_tfidf.shape)
print("Test TF-IDF Shape:", X_test_tfidf.shape)

Training TF-IDF Shape: (1600, 12058)
Validation TF-IDF Shape: (400, 12058)
Test TF-IDF Shape: (500, 12058)


In [8]:
# Train LinearSVC
svm_model = LinearSVC(
    C=1.0,
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

print("LinearSVC model trained successfully!")

LinearSVC model trained successfully!


In [9]:
# Make predictions on validation set
y_pred = svm_model.predict(X_val_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 1.0000


In [10]:
# Fit TF-IDF on the complete training data
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

X_full_tfidf = tfidf.fit_transform(X)
X_test_tfidf = tfidf.transform(X_test)

# Train LinearSVC on the full dataset
final_model = LinearSVC(
    C=1.0,
    random_state=42
)

final_model.fit(X_full_tfidf, y)

print("Final model trained on full dataset.")

Final model trained on full dataset.


In [11]:
# Predict on the test set
test_predictions = final_model.predict(X_test_tfidf)

# Create submission file
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_predictions
})

# Save submission
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\nsubmission.csv generated successfully!")

   id answer
0   1      A
1   2      B
2   3      B
3   4      E
4   5      C

submission.csv generated successfully!
